# 📦 Content-Based Filtering – Furniture Shop ML Service

Notebook này trình bày phương pháp **Content-Based Filtering** được triển khai trong `ml_service/app/services/recommendation_service.py`.

## Tổng quan
Hệ thống đề xuất sản phẩm dựa trên **nội dung** (tên, mô tả, danh mục, thương hiệu, tag, màu sắc, chất liệu) của chính sản phẩm.

### Hai chế độ hoạt động:
| Chế độ | Điều kiện | Mô tả |
|--------|-----------|-------|
| **Product-based** | có `target_product_id` | Tìm sản phẩm tương tự với sản phẩm đang xem |
| **User-based** | có `target_user_id` | Tạo user profile từ lịch sử → đề xuất phù hợp |
| **Fallback** | không có interaction | Xếp theo `sold_count + average_rating` |

## 1. Thiết lập môi trường

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', '-q'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')
print('✅ Thư viện đã sẵn sàng')

## 2. Nạp dữ liệu từ CSV export

In [ ]:
# Đọc dữ liệu từ các file CSV được export từ database
products_raw = pd.read_csv('furniture_shop.products.csv')
reviews_raw  = pd.read_csv('furniture_shop.reviews.csv')
orders_raw   = pd.read_csv('furniture_shop.orders.csv')
categories   = pd.read_csv('furniture_shop.categories.csv')
brands       = pd.read_csv('furniture_shop.brands.csv')

print(f'📦 Sản phẩm   : {len(products_raw)} bản ghi')
print(f'⭐ Đánh giá   : {len(reviews_raw)} bản ghi')
print(f'🛒 Đơn hàng   : {len(orders_raw)} bản ghi')
print(f'📂 Danh mục   : {len(categories)} bản ghi')
print(f'🏷️ Thương hiệu: {len(brands)} bản ghi')
products_raw.head(3)

## 3. Chuẩn bị dữ liệu – `data_prep_service.py` → `products_to_df()`

Hàm `products_to_df()` chuyển đổi raw product list thành DataFrame với trường **corpus** — feature text dùng cho TF-IDF.

```
corpus = name + description + category + brand + tags + colors + materials + image_count + has_3d
```

In [ ]:
# Tái hiện logic từ products_to_df() và interactions_to_df()

def build_products_df(products_raw, categories, brands):
    """Tái hiện products_to_df() từ data_prep_service.py"""
    # Map category và brand IDs
    cat_map   = dict(zip(categories['_id'].astype(str), categories['name']))
    brand_map = dict(zip(brands['_id'].astype(str), brands['name']))

    rows = []
    for _, p in products_raw.iterrows():
        pid          = str(p['_id'])
        category_name = cat_map.get(str(p.get('category', '')), '')
        brand_name    = brand_map.get(str(p.get('brand', '')), '')

        # Đếm số ảnh từ các cột images[*]
        img_cols    = [c for c in products_raw.columns if c.startswith('images[')]
        img_count   = int(p[img_cols].notna().sum())
        has_3d      = 1.0 if (pd.notna(p.get('model3DUrl','')) and str(p.get('model3DUrl','')) not in ('', 'nan')) else 0.0

        # Gom tag / color / material từ các cột dạng mảng
        def join_col(prefix):
            cols = [c for c in products_raw.columns if c.startswith(prefix)]
            vals = [str(p[c]) for c in cols if pd.notna(p[c]) and str(p[c]) != 'nan']
            return ' '.join(vals)

        # Tạo corpus (feature text cho TF-IDF)
        corpus = ' '.join([
            str(p.get('name', '')),
            str(p.get('description', '')),
            category_name, brand_name,
            join_col('tags['),
            join_col('colors['),
            join_col('materials['),
            f'image_count_{img_count}',
            'has_3d_model' if has_3d else 'no_3d_model'
        ])

        rows.append({
            'product_id'    : pid,
            'name'          : str(p.get('name', '')),
            'price'         : float(p.get('price', 0) or 0),
            'average_rating': float(p.get('averageRating', 0) or 0),
            'total_reviews' : float(p.get('totalReviews', 0) or 0),
            'sold_count'    : float(p.get('soldCount', 0) or 0),
            'stock'         : float(p.get('stock', 0) or 0),
            'image_count'   : float(img_count),
            'has_3d'        : has_3d,
            'corpus'        : corpus,
        })
    return pd.DataFrame(rows)

p_df = build_products_df(products_raw, categories, brands)
print(f'✅ Đã xây dựng xong DataFrame sản phẩm: {p_df.shape}')
p_df[['product_id','name','price','sold_count','average_rating','corpus']].head()

In [ ]:
# Xem corpus của một sản phẩm mẫu
print('📝 Corpus mẫu cho sản phẩm đầu tiên:')
print('='*60)
print(p_df.iloc[0]['corpus'])

## 4. TF-IDF Vectorization

**TF-IDF (Term Frequency–Inverse Document Frequency)**: mỗi sản phẩm được chuyển thành một vector số học từ trường `corpus`.

| Tham số | Giá trị | Ý nghĩa |
|---------|---------|----------|
| `stop_words` | `'english'` | Loại bỏ các từ phổ biến như "the", "and"... |
| `ngram_range` | `(1, 2)` | Sử dụng cả unigram và bigram |
| `max_features` | `5000` | Giới hạn 5000 đặc trưng |

In [ ]:
# Tái hiện logic TF-IDF từ content_based_recommendation()
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(p_df['corpus'])
print(f'📐 Kích thước ma trận TF-IDF: {tfidf_matrix.shape}')
print(f'   → {tfidf_matrix.shape[0]} sản phẩm × {tfidf_matrix.shape[1]} đặc trưng')

# Top features
feature_names = vectorizer.get_feature_names_out()
print(f'\n🔑 Ví dụ đặc trưng: {list(feature_names[:20])}')

## 5. Cosine Similarity Matrix

In [ ]:
# Tính cosine similarity giữa các sản phẩm
sim_matrix = cosine_similarity(tfidf_matrix)
product_ids = p_df['product_id'].tolist()
idx_by_pid  = {pid: i for i, pid in enumerate(product_ids)}

print(f'📊 Ma trận Similarity: {sim_matrix.shape}')

# Visualise ma trận similarity
sim_df = pd.DataFrame(sim_matrix,
                      index=p_df['name'].str[:20],
                      columns=p_df['name'].str[:20])

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(sim_df, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Cosine Similarity'})
ax.set_title('Ma trận Cosine Similarity giữa các Sản phẩm (TF-IDF Corpus)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Chế độ A: Đề xuất theo sản phẩm đang xem (target_product_id)

In [ ]:
def recommend_by_product(target_product_id: str, top_k: int = 5):
    """Tái hiện logic trong content_based_recommendation() khi có target_product_id"""
    if target_product_id not in idx_by_pid:
        print(f'❌ Không tìm thấy sản phẩm: {target_product_id}')
        return []

    target_idx = idx_by_pid[target_product_id]
    scores     = sim_matrix[target_idx]   # Vector similarity với tất cả SP
    rank       = np.argsort(scores)[::-1] # Sắp xếp từ cao → thấp

    results = []
    for idx in rank:
        if product_ids[idx] == target_product_id:
            continue  # Bỏ qua chính sản phẩm đó
        results.append({
            'product_id': product_ids[idx],
            'name'      : p_df.iloc[idx]['name'],
            'score'     : round(float(scores[idx]), 4),
            'price'     : p_df.iloc[idx]['price'],
        })
        if len(results) >= top_k:
            break
    return results

# Test với sản phẩm đầu tiên (Sofa Góc Lullaby)
target_pid = p_df.iloc[0]['product_id']
target_name = p_df.iloc[0]['name']
print(f'🔍 Đề xuất sản phẩm tương tự với: "{target_name}"\n')

recs = recommend_by_product(target_pid, top_k=5)
rec_df = pd.DataFrame(recs)
print(rec_df.to_string(index=False))

In [ ]:
# Visualise kết quả
rec_df_sorted = rec_df.sort_values('score')
colors = sns.color_palette('Blues_d', len(rec_df_sorted))

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(rec_df_sorted['name'].str[:30], rec_df_sorted['score'], color=colors)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
ax.set_xlabel('Cosine Similarity Score')
ax.set_title(f'Top 5 sản phẩm tương tự với "{target_name[:30]}" (Content-Based)', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Xây dựng User–Product Interactions → `interactions_to_df()`

Điểm tương tác (score) được tính từ:
- **Review**: `score += rating`
- **Order item**: `score += quantity × 1.5`  ← mua hàng có trọng số cao hơn

In [ ]:
def build_interactions_df(reviews_raw, orders_raw):
    """Tái hiện interactions_to_df() từ data_prep_service.py"""
    signals = defaultdict(float)

    # Tín hiệu từ reviews
    for _, r in reviews_raw.iterrows():
        uid = str(r['user'])
        pid = str(r['product'])
        rating = float(r.get('rating', 0) or 0)
        if uid and pid:
            signals[(uid, pid)] += rating

    # Tín hiệu từ orders (items)
    item_product_cols  = [c for c in orders_raw.columns if c.startswith('items[') and c.endswith('.product')]
    item_quantity_cols = [c for c in orders_raw.columns if c.startswith('items[') and c.endswith('.quantity')]

    for _, o in orders_raw.iterrows():
        uid = str(o['user'])
        for pc, qc in zip(item_product_cols, item_quantity_cols):
            pid = o.get(pc)
            qty = o.get(qc)
            if pd.notna(pid) and pd.notna(qty):
                signals[(uid, str(pid))] += float(qty) * 1.5  # trọng số mua hàng

    rows = [{'user_id': u, 'product_id': p, 'score': s}
            for (u, p), s in signals.items()]
    return pd.DataFrame(rows)

interactions = build_interactions_df(reviews_raw, orders_raw)
print(f'✅ Bảng tương tác: {interactions.shape}')
print(f'   Số users: {interactions["user_id"].nunique()}')
print(f'   Số products: {interactions["product_id"].nunique()}')
interactions.head(8)

## 8. Chế độ B: Đề xuất theo User Profile (target_user_id)

In [ ]:
def recommend_by_user(target_user_id: str, top_k: int = 5):
    """Tái hiện logic content_based_recommendation() khi có target_user_id"""
    user_items = interactions[interactions['user_id'] == target_user_id]['product_id'].tolist()
    if not user_items:
        print(f'⚠️ User {target_user_id} chưa có tương tác → Fallback theo popularity')
        return p_df.sort_values(['sold_count','average_rating'], ascending=False).head(top_k)[['product_id','name']].to_dict('records')

    user_indices = [idx_by_pid[p] for p in user_items if p in idx_by_pid]
    if not user_indices:
        return []

    # Profile = trung bình vector similarity của các SP đã tương tác
    profile = sim_matrix[user_indices].mean(axis=0)
    ranked  = np.argsort(profile)[::-1]
    seen    = set(user_items)

    results = []
    for idx in ranked:
        pid = product_ids[idx]
        if pid in seen:
            continue
        results.append({
            'product_id': pid,
            'name'      : p_df.iloc[idx]['name'],
            'score'     : round(float(profile[idx]), 4),
        })
        if len(results) >= top_k:
            break
    return results

# Chọn user đầu tiên trong dữ liệu
sample_user = interactions['user_id'].value_counts().index[0]
user_history = interactions[interactions['user_id'] == sample_user]['product_id'].tolist()
user_product_names = p_df[p_df['product_id'].isin(user_history)]['name'].tolist()

print(f'👤 User: {sample_user}')
print(f'📋 Lịch sử tương tác ({len(user_history)} sản phẩm):')
for n in user_product_names[:5]: print(f'   • {n}')

print(f'\n🎯 Đề xuất Content-Based cho user này:')
recs_user = recommend_by_user(sample_user, top_k=5)
pd.DataFrame(recs_user)

In [ ]:
# Visualise profile trung bình của user vs fallback popularity
# So sánh điểm similarity profile của SP đã thấy vs chưa thấy
user_indices_set = set(idx_by_pid.get(p) for p in user_history if p in idx_by_pid)
profile = sim_matrix[[i for i in user_indices_set]].mean(axis=0)

compare_df = pd.DataFrame({
    'name' : p_df['name'],
    'score': profile,
    'seen' : p_df['product_id'].isin(user_history)
})

fig, ax = plt.subplots(figsize=(12, 5))
colors_map = compare_df['seen'].map({True: '#e74c3c', False: '#2ecc71'})
bars = ax.bar(range(len(compare_df)), compare_df['score'], color=colors_map, edgecolor='white', linewidth=0.5)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Đã tương tác (bỏ qua)'),
                   Patch(facecolor='#2ecc71', label='Chưa tương tác (được đề xuất)')]
ax.legend(handles=legend_elements, fontsize=9)
ax.set_xticks(range(len(compare_df)))
ax.set_xticklabels(compare_df['name'].str[:18], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Profile Similarity Score')
ax.set_title(f'User Profile Similarity – Content-Based (user: {sample_user[:16]}...)', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Fallback: Popularity-based

In [ ]:
# Khi không có user interaction → fallback dùng sold_count + average_rating
popularity = p_df.copy()
popularity['popularity_score'] = popularity['sold_count'] + popularity['average_rating']
popularity = popularity.sort_values('popularity_score', ascending=False).head(5)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(popularity['name'].str[:25], popularity['sold_count'], color='#3498db')
axes[0].set_title('Top sản phẩm theo Sold Count', fontweight='bold')
axes[0].set_xlabel('Số lượng đã bán')

axes[1].barh(popularity['name'].str[:25], popularity['popularity_score'], color='#e67e22')
axes[1].set_title('Top sản phẩm theo Popularity Score\n(sold_count + average_rating)', fontweight='bold')
axes[1].set_xlabel('Điểm phổ biến')

plt.suptitle('Fallback: Popularity-Based Recommendation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

popularity[['name', 'sold_count', 'average_rating', 'popularity_score']]

## 10. Tóm tắt luồng xử lý

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║           Content-Based Filtering – Luồng Xử Lý                 ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. INPUT: products[]  →  products_to_df()                       ║
║           └─ Tạo trường corpus (name+desc+category+brand+...)    ║
║                                                                  ║
║  2. TfidfVectorizer (stop_words=english, ngram=(1,2), max=5000)  ║
║           └─ Mỗi product → 1 vector TF-IDF                      ║
║                                                                  ║
║  3. cosine_similarity(tfidf_matrix) → sim_matrix[N×N]            ║
║                                                                  ║
║  4. Routing:                                                     ║
║     ├─ target_product_id  → scores[target] → argsort → top_k    ║
║     ├─ target_user_id     → user_indices                         ║
║     │   └─ profile = sim_matrix[user_indices].mean(axis=0)       ║
║     │   └─ argsort(profile) → loại seen → top_k                 ║
║     └─ Fallback           → sold_count + average_rating          ║
║                                                                  ║
║  5. OUTPUT: [{product_id, name, score}, ...]                     ║
╚══════════════════════════════════════════════════════════════════╝
""")